# Train + compare, on Colab (trimmed)

This is a trimmed copy of `person_pseudolabels_test.ipynb`'s Sections 8-11, for running
**just the training + comparison** on Colab — Sections 1-7 (the pseudo-labeling pipeline
itself) already ran locally and their output (`run1_as_is` / `run2_pseudo_labeled` under
`experiments/merge_sim/`) is assumed to already be uploaded to your Drive, alongside
`yolo26n.pt`. Nothing here needs the raw datasets (`css-data`, `PPE Kit Detection`,
`ppe_detection_m`) at all.

**Before running:** make sure `experiments/merge_sim/run1_as_is`,
`experiments/merge_sim/run2_pseudo_labeled`, and `yolo26n.pt` have fully finished
uploading/syncing to your Drive — check with the `!ls` trick from earlier if unsure.


In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Check you actually have a GPU

Free Colab doesn't always default to one — if this prints `False`, go to
**Runtime > Change runtime type** and pick a GPU (e.g. T4), then re-run this cell before
continuing. Training on CPU here would be no faster than your laptop.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU assigned — Runtime > Change runtime type > pick a GPU, then re-run this cell.")

## Config — paths on Drive

Adjust `PROJECT_ROOT` below if your `final_project` folder is nested differently in your
Drive than `MyDrive/Colab Notebooks/final_project`.


In [ ]:
from pathlib import Path
import shutil

import yaml
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/final_project")  # <- adjust if nested differently

MERGE_ROOT = PROJECT_ROOT / "experiments" / "merge_sim"
RUN1_DIR = MERGE_ROOT / "run1_as_is"
RUN2_DIR = MERGE_ROOT / "run2_pseudo_labeled"

BASE_CHECKPOINT = PROJECT_ROOT / "yolo26n.pt"
TRAIN_EPOCHS = 40
IMGSZ = 640
BATCH = 16  # lower this (e.g. 8) if you hit a CUDA out-of-memory error

# Train onto Colab's own local disk, not straight onto the Drive mount — writing lots of
# small files directly to a Drive FUSE mount during training is a known way to make it
# slow or hang. Each run's results get copied to Drive right after it finishes instead,
# so nothing is lost even though /content itself is wiped when this runtime disconnects.
LOCAL_RUNS_DIR = Path("/content/runs/detect")

for p in (RUN1_DIR / "data.yaml", RUN2_DIR / "data.yaml", BASE_CHECKPOINT):
    assert p.exists(), f"missing {p} — check PROJECT_ROOT above, and that the upload to Drive finished"

print("all good — ready to train")

## Fix each `data.yaml`'s `path`

Both files still have the `path:` they were built with locally (your Mac's path) — that
needs to point at wherever Drive is mounted here instead, or Ultralytics won't find the
images even though the folder is right there.


In [ ]:
for run_dir in (RUN1_DIR, RUN2_DIR):
    yaml_path = run_dir / "data.yaml"
    cfg = yaml.safe_load(yaml_path.read_text())
    cfg["path"] = str(run_dir.resolve())
    yaml_path.write_text(yaml.safe_dump(cfg))
    print(f"fixed {yaml_path} -> path: {cfg['path']}")

run1_yaml = RUN1_DIR / "data.yaml"
run2_yaml = RUN2_DIR / "data.yaml"

## Train both models

Same `BASE_CHECKPOINT` (`yolo26n.pt`), same epochs/image size/batch for both — the only
difference between the two calls is which `data.yaml` they point at, so any gap in the
results is attributable to the Person boxes, not the setup. `device` is left unset so
Ultralytics picks up the GPU automatically once one's assigned (checked above).


In [ ]:
run1_model = YOLO(str(BASE_CHECKPOINT))
run1_train_results = run1_model.train(
    data=str(run1_yaml),
    epochs=TRAIN_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(LOCAL_RUNS_DIR),
    name="merge_sim_run1_as_is",
    exist_ok=True,
)

# copy immediately, before starting Run 2, so this result is safe even if Run 2 fails
shutil.copytree(
    LOCAL_RUNS_DIR / "merge_sim_run1_as_is",
    PROJECT_ROOT / "runs" / "detect" / "merge_sim_run1_as_is",
    dirs_exist_ok=True,
)
print("copied Run 1 results to Drive")

In [ ]:
run2_model = YOLO(str(BASE_CHECKPOINT))
run2_train_results = run2_model.train(
    data=str(run2_yaml),
    epochs=TRAIN_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(LOCAL_RUNS_DIR),
    name="merge_sim_run2_pseudo_labeled",
    exist_ok=True,
)

shutil.copytree(
    LOCAL_RUNS_DIR / "merge_sim_run2_pseudo_labeled",
    PROJECT_ROOT / "runs" / "detect" / "merge_sim_run2_pseudo_labeled",
    dirs_exist_ok=True,
)
print("copied Run 2 results to Drive")

## Compare results

Both runs share the same unified class list they were built with, so — unlike the
per-class table on the app's Model Performance page — these two are directly comparable
class-for-class, including Person.


In [ ]:
RUN1_RESULTS_DIR = LOCAL_RUNS_DIR / "merge_sim_run1_as_is"
RUN2_RESULTS_DIR = LOCAL_RUNS_DIR / "merge_sim_run2_pseudo_labeled"

def final_row(results_dir):
    df = pd.read_csv(results_dir / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    return {
        "epochs_run": int(last["epoch"]),
        "precision": round(last["metrics/precision(B)"], 3),
        "recall": round(last["metrics/recall(B)"], 3),
        "mAP50": round(last["metrics/mAP50(B)"], 3),
        "mAP50-95": round(last["metrics/mAP50-95(B)"], 3),
    }

comparison = pd.DataFrame({
    "Run 1 — as-is": final_row(RUN1_RESULTS_DIR),
    "Run 2 — pseudo-labeled": final_row(RUN2_RESULTS_DIR),
}).T
comparison.index.name = "run"
print(comparison)

In [ ]:
# --- confusion matrices side by side, same class axes on both ---------------------
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, results_dir, title in (
    (axes[0], RUN1_RESULTS_DIR, "Run 1 — as-is"),
    (axes[1], RUN2_RESULTS_DIR, "Run 2 — pseudo-labeled"),
):
    cm_path = results_dir / "confusion_matrix_normalized.png"
    if not cm_path.exists():
        cm_path = results_dir / "confusion_matrix.png"
    ax.imshow(mpimg.imread(cm_path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Reading this

- **This is a comparability test, not a verdict.** Check the printed `epochs_run` counts
  above — if either run stopped early (Ultralytics' built-in patience-based early
  stopping), that's worth noting alongside the numbers, not just the final scores.
- **Where to expect a difference, if there is one:** Person precision/recall, and the
  Person row/column of the confusion matrix specifically — that's the only supervision
  that changed between the two runs.
- **Everything's now saved to your Drive** under
  `final_project/runs/detect/merge_sim_run1_as_is` and `merge_sim_run2_pseudo_labeled` —
  safe to close this Colab session, the results won't disappear with it.
- **Next step if Run 2 looks better:** back on the original notebook, rerun Section 5 over
  more of `ppe_detection_m` (bigger `SAMPLE_N`, or loop all three splits), rebuild the
  samples in Section 8 with the larger pool, and consider a longer training run before
  treating any resulting weights as a real candidate.
